In [3]:
%pip install pandas
import pandas as pd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 26.0.1
[notice] To update, run: C:\Users\neome\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
stats = pd.read_csv('data/advanced_stats_all.csv')
salary = pd.read_csv('data/player_salary.csv')

In [ ]:
stats_clean = stats.copy()

# Drop columns we do not need for modeling
stats_clean = stats_clean.drop(columns=["Rk", "Player-additional"], errors="ignore")

# dropping duplicate player rows and empty columns
stats_clean = stats_clean.drop_duplicates(subset=["Player", "Team"], keep="first")
stats_clean = stats_clean.dropna(axis=1, how="all")

# Fill awards with empty string so it is consistent text data
if "Awards" in stats_clean.columns:
    stats_clean["Awards"] = stats_clean["Awards"].fillna("")

for col in ["Player", "Team", "Pos"]:
    if col in stats_clean.columns:
        stats_clean[col] = stats_clean[col].astype(str).str.strip()

salary_clean = salary.copy()

# Rename known placeholder columns first
salary_clean = salary_clean.rename(
    columns={
        "Unnamed: 0": "Rk",
        "Unnamed: 1": "Player",
        "Unnamed: 2": "Team",
    }
)

salary_clean = salary_clean[salary_clean["Rk"].astype(str) != "Rk"].copy()

# Rename salary columns to readable season names if present
season_map = {
    "Salary": "2025-26",
    "Salary.1": "2026-27",
    "Salary.2": "2027-28",
    "Salary.3": "2028-29",
    "Salary.4": "2029-30",
    "Salary.5": "2030-31",
    "Unnamed: 9": "Guaranteed",
    "-additional": "player_id",
}
salary_clean = salary_clean.rename(columns={k: v for k, v in season_map.items() if k in salary_clean.columns})

# keep only useful columns if they exist
keep_cols = [
    "Rk", "Player", "Team",
    "2025-26", "2026-27", "2027-28", "2028-29", "2029-30", "2030-31",
    "Guaranteed", "player_id",
]
salary_clean = salary_clean[[c for c in keep_cols if c in salary_clean.columns]]

# Parse currency strings to numeric
money_cols = [c for c in salary_clean.columns if c.startswith("20") or c == "Guaranteed"]
for col in money_cols:
    salary_clean[col] = (
        salary_clean[col]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace({"": pd.NA})
    )
    salary_clean[col] = pd.to_numeric(salary_clean[col], errors="coerce")

# making the rank a number
if "Rk" in salary_clean.columns:
    salary_clean["Rk"] = pd.to_numeric(salary_clean["Rk"], errors="coerce")

# make the names text and remove empty names
if "Player" in salary_clean.columns:
    salary_clean["Player"] = salary_clean["Player"].astype("string").str.strip()
    salary_clean = salary_clean[salary_clean["Player"].notna() & (salary_clean["Player"] != "")]

# Basic text cleanup without converting nulls to literal "nan"
for col in ["Team", "player_id"]:
    if col in salary_clean.columns:
        salary_clean[col] = salary_clean[col].astype("string").str.strip()


stats_clean.to_csv("data/advanced_stats_clean.csv", index=False)
salary_clean.to_csv("data/player_salary_clean.csv", index=False)

print("Advanced stats cleaned shape:", stats_clean.shape)
print("Salary cleaned shape:", salary_clean.shape)
print("\nAdvanced stats preview:")
print(stats_clean.head())
print("\nSalary preview:")
print(salary_clean.head())

Advanced stats cleaned shape: (5441, 59)
Salary cleaned shape: (530, 11)

Advanced stats preview:
            Player   Age Team Pos     G    GS      MP   PER    TS%   3PAr  \
0     Kevin Durant  21.0  OKC  SF  82.0  82.0  3239.0  26.2  0.607  0.210   
1   Andre Iguodala  26.0  PHI  SF  82.0  82.0  3193.0  17.8  0.535  0.271   
2         Rudy Gay  23.0  MEM  SF  80.0  80.0  3175.0  16.2  0.535  0.157   
3  Stephen Jackson  31.0  2TM  SG  81.0  81.0  3129.0  15.6  0.518  0.277   
4  Stephen Jackson  31.0  GSW  PF   9.0   9.0   300.0  14.5  0.499  0.301   

   ...  19.6  3.7  2.0  5.7  .090  0.4  -0.9  -0.5  1.2.1  bridgmi01  
0  ...   NaN  NaN  NaN  NaN   NaN  NaN   NaN   NaN    NaN        NaN  
1  ...   NaN  NaN  NaN  NaN   NaN  NaN   NaN   NaN    NaN        NaN  
2  ...   NaN  NaN  NaN  NaN   NaN  NaN   NaN   NaN    NaN        NaN  
3  ...   NaN  NaN  NaN  NaN   NaN  NaN   NaN   NaN    NaN        NaN  
4  ...   NaN  NaN  NaN  NaN   NaN  NaN   NaN   NaN    NaN        NaN  

[5 rows x 59

: 